In [9]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "bigscience/bloom-3b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.01G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

In [21]:
ds = load_dataset("tatsu-lab/alpaca")

In [10]:
lora_config = LoraConfig(r=50, target_modules=['query_key_value', 'dense_h_to_4h', 'dense_4h_to_h'])
model = get_peft_model(model, lora_config)

In [13]:
model.print_trainable_parameters()

trainable params: 53,760,000 || all params: 3,056,317,440 || trainable%: 1.7590


In [26]:
ds['train']['text'][0]

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'

In [27]:
template = tokenizer('### Response:')
template

{'input_ids': [105311, 66673, 29], 'attention_mask': [1, 1, 1]}

In [32]:
tokenizer(ds['train']['text'][0])

{'input_ids': [111757, 632, 660, 54103, 861, 63808, 267, 20165, 17, 66828, 267, 12427, 861, 156788, 115739, 368, 8821, 6149, 105311, 182924, 29, 189, 119158, 8603, 63211, 613, 135576, 70349, 6149, 105311, 66673, 29, 189, 20, 17, 40, 278, 267, 123002, 46949, 530, 5219, 11097, 427, 13756, 95063, 461, 56326, 530, 140550, 10209, 21, 17, 230539, 107237, 427, 11874, 2632, 12364, 16153, 530, 16045, 10209, 22, 17, 12018, 17123, 35237, 530, 46944, 267, 37552, 35237, 74868, 17], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [33]:
input_ids = tokenizer(ds['train']['text'][0])['input_ids']

In [34]:
labels = input_ids

In [35]:
print(labels)

[111757, 632, 660, 54103, 861, 63808, 267, 20165, 17, 66828, 267, 12427, 861, 156788, 115739, 368, 8821, 6149, 105311, 182924, 29, 189, 119158, 8603, 63211, 613, 135576, 70349, 6149, 105311, 66673, 29, 189, 20, 17, 40, 278, 267, 123002, 46949, 530, 5219, 11097, 427, 13756, 95063, 461, 56326, 530, 140550, 10209, 21, 17, 230539, 107237, 427, 11874, 2632, 12364, 16153, 530, 16045, 10209, 22, 17, 12018, 17123, 35237, 530, 46944, 267, 37552, 35237, 74868, 17]


In [ ]:
r = {'input_ids':[], 'labels':[], 'attention_mask':[]}
for i in range(len(input_ids)-1):
  r['input_ids']